# Audit Design Simulation

## Hierarchical Generative Model

For each replicate $r = 1, \dots, R$ and repository $k = 1, \dots, K$:

### Repository level

$$
\begin{aligned}
n_k &\sim \text{LogNormal}(\mu_n, \sigma_n^2),\quad n_k \geq n_{\min} \\
\rho^{\text{corr}}_k &\sim \text{Beta}(a_{\text{corr}}, b_{\text{corr}}) \\
\rho^{\text{intr}}_k &\sim \text{Beta}(a_{\text{intr}}, b_{\text{intr}}) \\
N_k &= \max\bigl(1,\; \lfloor n_k \cdot \rho^{\text{corr}}_k \cdot \rho^{\text{intr}}_k \rceil\bigr) \\
\mu^{\text{repo}}_k &\sim \mathcal{N}(\mu_p, \sigma_p^2),\quad \text{clipped to }[0.001, 0.999] \\
s_k &\sim \mathcal{N}(0, \sigma_s^2) \\
\boldsymbol{\pi}_k &\sim \text{Dirichlet}(\boldsymbol{\alpha}_{\text{mix}})
\end{aligned}
$$

### Change level (for $i = 1, \dots, N_k$)

$$
\begin{aligned}
g_{ki} &\sim \text{Categorical}(\boldsymbol{\pi}_k) \\
\tilde{p}_{ki} &\sim \text{Beta}\bigl(a(\mu_{g_{ki}}, \sigma_g),\; b(\mu_{g_{ki}}, \sigma_g)\bigr) \\
p_{ki} &= \sigma\!\left(\text{logit}\bigl(\tfrac{1}{2}\tilde{p}_{ki} + \tfrac{1}{2}\mu^{\text{repo}}_k\bigr) + s_k\right) \\
Y_{ki} &\sim \text{Bernoulli}(p_{ki}) \\
\rho_{ki} &= \text{clip}\bigl(m(p_{ki}, g_{ki}) + \varepsilon_{ki},\; 0,\; 1\bigr),\quad \varepsilon_{ki} \sim \mathcal{N}(0, \sigma^2_{\rho, g_{ki}})
\end{aligned}
$$

with signal function $m(p, g)$ depending on scenario:

| Scenario | $m(p, g)$ |
|---|---|
| Monotonic | $1 - p$ |
| Non-Monotonic | category-specific constant $\in \{0.9, 0.1, 0.9\}$ |
| Uninformative | $U \sim \text{Uniform}(0,1)$ |

$(a, b)$ are the standard moment-matched Beta parameters from $(\mu, \sigma)$.

## Sampling and Estimation

Strata: $S_h = \{i : \rho_i \in [(h-1)/J, h/J)\}$ for $h = 1, \dots, J$.

Per design, draw audit indices $\mathcal{I} \subset \{1, \dots, N\}$ of size $\leq n$:

$$
n_h = \begin{cases}
\lfloor n \cdot |S_h|/N \rceil & \text{proportional} \\
\lfloor n / J \rceil & \text{equal} \\
\lambda \cdot n_h^{\text{prop}} + (1-\lambda) \cdot n_h^{\text{equal}} & \text{hybrid}(\lambda)
\end{cases}
$$

Beta-mixture posterior over $M$ Monte Carlo draws:

$$
\alpha_h^{(m)} \sim \text{Beta}\bigl(1 + y_h,\; 1 + n_h - y_h\bigr),\quad
\alpha^{(m)} = \sum_h \tfrac{|S_h|}{N}\, \alpha_h^{(m)}
$$

with $y_h = \sum_{i \in \mathcal{I} \cap S_h} Y_i$. Point estimate $\hat{\alpha} = \overline{\alpha^{(m)}}$; CI from $[2.5\%, 97.5\%]$ quantiles.

## Configuration

| Symbol | Values |
|---|---|
| $K$ | $\{5, 10, 20\}$ |
| $n$ | $\{20, 40, 60, 80, 100\}$ |
| $J$ | $\{3, 5\}$ |
| $\lambda$ | $\{0.25, 0.50, 0.75\}$ (hybrid only) |
| $\mu_p$ | $\{0.15, 0.30, 0.50\}$ |
| $\sigma_p$ | $\{0.05, 0.10, 0.15\}$ |
| $\sigma_g$ | $\{0.05, 0.10, 0.15\}$ |
| $R$ | $200$ |

Fixed: $\mu_n = 3276$, $\sigma_n = 2960$, $n_{\min} = 100$, $\sigma_s = 0.30$, scenario-specific $\boldsymbol{\alpha}_{\text{mix}}, \boldsymbol{\mu}_g, \boldsymbol{\sigma}_\rho$.

In [ ]:
from abc import ABC, abstractmethod
import numpy as np
from scipy.stats import beta
from scipy.special import expit, logit


class AuditDesign(ABC):
    name = None

    @staticmethod
    def strata(rho, J):
        bins = np.linspace(0, 1, J + 1)
        strata = np.digitize(rho, bins, right=False) - 1
        return np.clip(strata, 0, J - 1)

    @abstractmethod
    def sample(self, strata, n_audit, rng):
        pass

    def estimate(self, true_labels, idx, strata, rng, n_draws=5000):
        N = len(strata)

        sampled_labels = true_labels[idx]
        sampled_strata = strata[idx]

        alpha_draws = np.zeros(n_draws)
        estimate = 0.0

        for h in np.unique(strata):
            pop_mask = strata == h
            samp_mask = sampled_strata == h

            W_h = pop_mask.sum() / N
            n_h = samp_mask.sum()
            y_h = sampled_labels[samp_mask].sum()

            draws_h = rng.beta(
                1 + y_h,
                1 + n_h - y_h,
                size=n_draws,
            )

            alpha_draws += W_h * draws_h

        estimate = alpha_draws.mean()

        ci_low, ci_high = np.quantile(alpha_draws, [0.025, 0.975])

        return estimate, ci_low, ci_high


class SimpleRandomSampling(AuditDesign):
    name = "simple"

    def sample(self, strata, n_audit, rng):
        N = len(strata)
        return rng.choice(N, size=min(n_audit, N), replace=False)


class ProportionalStratifiedSampling(AuditDesign):
    name = "proportional"

    def sample(self, strata, n_audit, rng):
        N = len(strata)
        unique, counts = np.unique(strata, return_counts=True)

        n_total = min(n_audit, N)
        props = counts / N
        raw = props * n_total
        n_per = np.floor(raw).astype(int)

        remainder = n_total - n_per.sum()
        if remainder > 0:
            order = np.argsort(-(raw - n_per))
            for i in order[:remainder]:
                n_per[i] += 1

        idx = []
        for s, n_s in zip(unique, n_per):
            if n_s <= 0:
                continue

            pool = np.where(strata == s)[0]
            n_draw = min(n_s, len(pool))

            if n_draw > 0:
                idx.append(rng.choice(pool, size=n_draw, replace=False))

        return np.concatenate(idx)


class EqualAllocationStratifiedSampling(AuditDesign):
    name = "equal"

    def sample(self, strata, n_audit, rng):
        unique = np.unique(strata)
        n_total = min(n_audit, len(strata))

        J_actual = len(unique)
        base = n_total // J_actual
        remainder = n_total - base * J_actual

        idx = []
        for i, s in enumerate(unique):
            pool = np.where(strata == s)[0]
            n_s = base + (1 if i < remainder else 0)
            n_draw = min(n_s, len(pool))

            if n_draw > 0:
                idx.append(rng.choice(pool, size=n_draw, replace=False))

        return np.concatenate(idx)


class HybridStratifiedSampling(AuditDesign):
    name = "hybrid"

    def __init__(self, proportional_fraction=0.75):
        self.proportional_fraction = proportional_fraction

    def sample(self, strata, n_audit, rng):
        N = len(strata)
        unique, counts = np.unique(strata, return_counts=True)

        n_total = min(n_audit, N)

        n_prop_total = int(round(self.proportional_fraction * n_total))
        n_equal_total = n_total - n_prop_total

        # proportional component
        raw_prop = counts / N * n_prop_total
        n_prop = np.floor(raw_prop).astype(int)

        rest = n_prop_total - n_prop.sum()
        if rest > 0:
            order = np.argsort(-(raw_prop - n_prop))
            for i in order[:rest]:
                n_prop[i] += 1

        # equal top-up component
        base = n_equal_total // len(unique)
        remainder = n_equal_total - base * len(unique)
        n_equal = np.array(
            [base + (1 if i < remainder else 0) for i in range(len(unique))]
        )

        # final allocation
        n_per = n_prop + n_equal

        idx = []

        for s, n_s in zip(unique, n_per):
            if n_s <= 0:
                continue

            pool = np.where(strata == s)[0]
            n_draw = min(n_s, len(pool))

            if n_draw > 0:
                chosen = rng.choice(pool, size=n_draw, replace=False)
                idx.append(chosen)

        return np.concatenate(idx)


class SimulationWorld:

    def __init__(
        self,
        repository_error_rate_mean,
        repository_error_rate_sd,
        change_error_sd,
        n_repositories,
        rng,
        repository_changes_mean,
        repository_changes_sd,
        repository_changes_min,
        repository_correction_rate_mean,
        repository_correction_rate_sd,
        repository_intrinsic_rate_mean,
        repository_intrinsic_rate_sd,
        repository_error_shock_sd,
        change_mix_alpha,
        change_error_rate,
        change_agreement_mode,
        change_agreement_noise,
        bimodal_split=None,
    ):
        self.repository_error_rate_mean = repository_error_rate_mean
        self.repository_error_rate_sd = repository_error_rate_sd
        self.repository_error_shock_sd = repository_error_shock_sd

        self.change_error_sd = change_error_sd
        self.n_repositories = n_repositories
        self.rng = rng

        self.repository_changes_mean = repository_changes_mean
        self.repository_changes_sd = repository_changes_sd
        self.repository_changes_min = repository_changes_min

        self.a_correction, self.b_correction = self._beta_params(
            repository_correction_rate_mean,
            repository_correction_rate_sd,
        )
        self.a_intrinsic, self.b_intrinsic = self._beta_params(
            repository_intrinsic_rate_mean,
            repository_intrinsic_rate_sd,
        )

        self.change_mix_alpha = np.array(change_mix_alpha)
        self.change_error_rate = np.array(change_error_rate)
        self.change_agreement_mode = np.array(change_agreement_mode)
        self.change_agreement_noise = np.array(change_agreement_noise)

        self.bimodal_split = bimodal_split  # NEW

    @staticmethod
    def _beta_params(mean, sd):
        var = sd**2
        max_var = mean * (1 - mean)

        if var >= max_var:
            var = max_var * 0.9

        common = mean * (1 - mean) / var - 1
        return mean * common, (1 - mean) * common

    def generate(self):
        pool_per_repository = self._generate_repository_pool_sizes()
        repository_ids = self._generate_repository_ids(pool_per_repository)
        changes = self._generate_changes(pool_per_repository)
        p_error = self._generate_error_probabilities(pool_per_repository, changes)

        true_errors = self._realize_errors(p_error)
        agreement = self._generate_agreement(p_error, changes)

        return {
            "true_labels": true_errors,
            "alpha_true": true_errors.mean(),
            "rho": agreement,
            "agreement": agreement,
            "changes": changes,
            "repository_ids": repository_ids,
            "p_error": p_error,
        }

    def _sample_changes_per_repository(self):
        var = self.repository_changes_sd**2
        mu = np.log(
            self.repository_changes_mean**2
            / np.sqrt(var + self.repository_changes_mean**2)
        )
        sigma_ln = np.sqrt(np.log(1 + var / self.repository_changes_mean**2))

        samples = []
        while len(samples) < self.n_repositories:
            x = self.rng.lognormal(mu, sigma_ln, size=self.n_repositories * 5)
            x = x[x >= self.repository_changes_min]
            samples.extend(x.tolist())

        return np.array(samples[: self.n_repositories]).astype(int)

    def _generate_repository_pool_sizes(self):
        changes_per_repository = self._sample_changes_per_repository()

        correction_rate = self.rng.beta(
            self.a_correction,
            self.b_correction,
            self.n_repositories,
        )
        intrinsic_rate = self.rng.beta(
            self.a_intrinsic,
            self.b_intrinsic,
            self.n_repositories,
        )

        return np.maximum(
            1,
            np.round(changes_per_repository * correction_rate * intrinsic_rate).astype(
                int
            ),
        )

    def _generate_repository_ids(self, pool_per_repository):
        return np.repeat(np.arange(self.n_repositories), pool_per_repository)

    def _generate_changes(self, pool_per_repository):
        changes = []

        for n_items in pool_per_repository:
            repository_change_mix = self.rng.dirichlet(self.change_mix_alpha)

            repository_changes = self.rng.choice(
                len(repository_change_mix),
                size=n_items,
                p=repository_change_mix,
            )

            changes.append(repository_changes)

        return np.concatenate(changes)

    def _generate_repository_means(self):
        if self.bimodal_split is None:
            return np.clip(
                self.rng.normal(
                    self.repository_error_rate_mean,
                    self.repository_error_rate_sd,
                    self.n_repositories,
                ),
                0.001,
                0.999,
            )

        mean_low, mean_high, share_low = self.bimodal_split
        K_low = int(round(self.n_repositories * share_low))
        K_high = self.n_repositories - K_low

        rates_low = self.rng.normal(mean_low, self.repository_error_rate_sd, K_low)
        rates_high = self.rng.normal(mean_high, self.repository_error_rate_sd, K_high)

        rates = np.concatenate([rates_low, rates_high])
        self.rng.shuffle(rates)

        return np.clip(rates, 0.001, 0.999)

    def _generate_error_probabilities(self, pool_per_repository, changes):
        p_error = np.empty(pool_per_repository.sum())

        repository_error_rate = self._generate_repository_means()

        start = 0

        for k, n_items in enumerate(pool_per_repository):
            end = start + n_items
            repository_changes = changes[start:end]

            change_mean = self.change_error_rate[repository_changes]

            if self.change_error_sd > 0:
                base_p = np.empty(n_items)

                for change in np.unique(repository_changes):
                    mask = repository_changes == change
                    mean = change_mean[mask][0]

                    a_w, b_w = self._beta_params(mean, self.change_error_sd)
                    base_p[mask] = self.rng.beta(a_w, b_w, mask.sum())
            else:
                base_p = change_mean.copy()

            repository_level = repository_error_rate[k]
            base_p = 0.5 * base_p + 0.5 * repository_level

            if self.repository_error_shock_sd > 0:
                shock = self.rng.normal(0, self.repository_error_shock_sd)
                base_p = expit(logit(np.clip(base_p, 0.001, 0.999)) + shock)

            p_error[start:end] = np.clip(base_p, 0.001, 0.999)
            start = end

        return p_error

    def _realize_errors(self, p_error):
        return (self.rng.random(len(p_error)) < p_error).astype(int)

    def _generate_agreement(self, p_error, changes):
        agreement = np.empty(len(p_error))

        for change in np.unique(changes):
            mask = changes == change
            mode = self.change_agreement_mode[change]
            noise = self.change_agreement_noise[change]

            if mode == "inverse_error":
                signal = 1 - p_error[mask]
            elif mode == "error":
                signal = p_error[mask]
            elif mode == "high":
                signal = np.full(mask.sum(), 0.90)
            elif mode == "low":
                signal = np.full(mask.sum(), 0.10)
            elif mode == "random":
                signal = self.rng.random(mask.sum())
            else:
                raise ValueError(f"Unknown agreement mode: {mode}")

            agreement[mask] = signal + self.rng.normal(
                0,
                noise,
                size=mask.sum(),
            )

        return np.clip(agreement, 0, 1)

In [ ]:
import hashlib
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from tqdm import tqdm
import itertools

N_REPLICATES = 200

designs = [
    ("proportional", 3, None),
    ("proportional", 5, None),
    ("equal", 3, None),
    ("equal", 5, None),
    ("hybrid", 3, 0.25),
    ("hybrid", 3, 0.50),
    ("hybrid", 3, 0.75),
    ("hybrid", 5, 0.25),
    ("hybrid", 5, 0.50),
    ("hybrid", 5, 0.75),
]

scenarios = {
    "Monotonic": {
        # two change categories: common (well-behaved) vs rare (noisy)
        "change_mix_alpha": (8, 2),
        # baseline error rate per change category
        "change_error_rate": (0.10, 0.40),
        # agreement signal: roughly inverse error, but noisy for rare cases
        "change_agreement_mode": ("inverse_error", "inverse_error"),
        "change_agreement_noise": (0.05, 0.50),
    },
    "Non-Monotonic": {
        # three categories: easy / ambiguous / systematic failure
        "change_mix_alpha": (7, 2, 1),
        # error rate per category
        "change_error_rate": (0.10, 0.35, 0.55),
        # agreement does not always track correctness
        "change_agreement_mode": ("high", "low", "high"),
        "change_agreement_noise": (0.05, 0.20, 0.05),
    },
    "Uninformative": {
        # different error rates exist, but agreement does not see them
        "change_mix_alpha": (7, 2, 1),
        "change_error_rate": (0.10, 0.35, 0.55),
        "change_agreement_mode": ("random", "random", "random"),
        "change_agreement_noise": (0.00, 0.00, 0.00),
    },
}

setups = list(
    itertools.product(
        [5, 10, 20],  # n_repositories
        [20, 40, 60, 80, 100],  # n_audit
        [0.15, 0.30, 0.50],  # repository_error_rate_mean
        [0.05, 0.10, 0.15],  # repository_error_rate_sd
        [0.05, 0.10, 0.15],  # change_error_sd
        list(scenarios.keys()),
        designs,
    )
)

print(f"Setups: {len(setups)}")
print(f"Total worlds: {len(setups) * N_REPLICATES:,}")


def compute_rho_curve_metrics(true_labels, rho, idx, J, fallback_alpha):
    bins = np.linspace(0, 1, J + 1)
    strata = np.digitize(rho, bins, right=False) - 1
    strata = np.clip(strata, 0, J - 1)

    sampled_indicator = np.zeros(len(strata), dtype=bool)
    sampled_indicator[idx] = True

    weighted_mse = 0.0
    weight_sum = 0.0

    errors = []
    weighted_errors = []

    true_curve = []
    est_curve = []

    for h in range(J):
        pop_mask = strata == h
        samp_mask = sampled_indicator & pop_mask

        N_h = pop_mask.sum()
        n_h = samp_mask.sum()

        if N_h == 0:
            continue

        true_rate = true_labels[pop_mask].mean()

        if n_h > 0:
            est_rate = (true_labels[samp_mask].sum() + 1) / (n_h + 2)
        else:
            est_rate = fallback_alpha

        W_h = N_h / len(strata)
        error_h = est_rate - true_rate

        weighted_mse += W_h * error_h**2
        weight_sum += W_h

        errors.append(error_h)
        weighted_errors.append(W_h * error_h)

        true_curve.append(true_rate)
        est_curve.append(est_rate)

    rho_curve_mse = weighted_mse / weight_sum if weight_sum > 0 else np.nan

    errors = np.array(errors)
    weighted_errors = np.array(weighted_errors)

    rho_curve_bias = np.mean(errors) if len(errors) > 0 else np.nan
    rho_curve_abs_bias = np.mean(np.abs(errors)) if len(errors) > 0 else np.nan
    rho_curve_variance = np.var(errors) if len(errors) > 1 else np.nan

    true_curve = np.array(true_curve)
    est_curve = np.array(est_curve)

    if len(true_curve) > 1 and np.std(true_curve) > 0 and np.std(est_curve) > 0:
        rho_curve_corr = np.corrcoef(true_curve, est_curve)[0, 1]
    else:
        rho_curve_corr = np.nan

    return (
        rho_curve_mse,
        rho_curve_corr,
        rho_curve_bias,
        rho_curve_abs_bias,
        rho_curve_variance,
    )


def simulate_design(
    repository_error_rate_mean,
    repository_error_rate_sd,
    change_error_sd,
    rho_scenario,
    n_repositories,
    n_audit,
    n_replicates,
    design,
    J,
    lambda_mix,
    bimodal_split,
    rng,
):
    if design == "hybrid":
        design_obj = HybridStratifiedSampling(proportional_fraction=lambda_mix)
    else:
        design_obj = {
            "simple": SimpleRandomSampling,
            "proportional": ProportionalStratifiedSampling,
            "equal": EqualAllocationStratifiedSampling,
        }[design]()

    estimates = np.empty(n_replicates)
    alpha_trues = np.empty(n_replicates)
    ci_widths = np.empty(n_replicates)
    covered = np.empty(n_replicates, dtype=bool)
    pool_totals = np.empty(n_replicates)

    rho_error_corrs = np.empty(n_replicates)
    rho_curve_mses = np.empty(n_replicates)
    rho_curve_corrs = np.empty(n_replicates)

    rho_curve_biases = np.empty(n_replicates)
    rho_curve_abs_biases = np.empty(n_replicates)
    rho_curve_variances = np.empty(n_replicates)

    scenario_kwargs = scenarios[rho_scenario]

    for r in range(n_replicates):
        world = SimulationWorld(
            repository_error_rate_mean=repository_error_rate_mean,
            repository_error_rate_sd=repository_error_rate_sd,
            change_error_sd=change_error_sd,
            n_repositories=n_repositories,
            rng=rng,
            # updated empirical scale
            repository_changes_mean=3276,
            repository_changes_sd=2960,
            repository_changes_min=100,
            # updated filtering (fewer corrections)
            repository_correction_rate_mean=0.10,
            repository_correction_rate_sd=0.033,
            # intrinsic slightly tighter
            repository_intrinsic_rate_mean=0.60,
            repository_intrinsic_rate_sd=0.084,
            repository_error_shock_sd=0.30,
            bimodal_split=None,
            # keep your scenario
            **scenario_kwargs,
        )

        data = world.generate()

        true_labels = data["true_labels"]
        alpha_true = data["alpha_true"]
        rho = data["rho"]

        strata = design_obj.strata(rho, J)

        idx = design_obj.sample(
            strata=strata,
            n_audit=n_audit,
            rng=rng,
        )

        estimate, ci_low, ci_high = design_obj.estimate(
            true_labels=true_labels,
            idx=idx,
            strata=strata,
            rng=rng,
        )

        estimates[r] = estimate
        alpha_trues[r] = alpha_true
        ci_widths[r] = ci_high - ci_low
        covered[r] = ci_low <= alpha_true <= ci_high
        pool_totals[r] = len(true_labels)

        if np.std(rho) > 0 and np.std(true_labels) > 0:
            rho_error_corrs[r] = np.corrcoef(rho, true_labels)[0, 1]
        else:
            rho_error_corrs[r] = np.nan

        (
            rho_curve_mses[r],
            rho_curve_corrs[r],
            rho_curve_biases[r],
            rho_curve_abs_biases[r],
            rho_curve_variances[r],
        ) = compute_rho_curve_metrics(
            true_labels=true_labels,
            rho=rho,
            idx=idx,
            J=J,
            fallback_alpha=estimate,
        )

    rho_error_corr_value = (
        float(np.nanmedian(rho_error_corrs))
        if np.any(~np.isnan(rho_error_corrs))
        else np.nan
    )

    rho_curve_mse_value = (
        float(np.nanmedian(rho_curve_mses))
        if np.any(~np.isnan(rho_curve_mses))
        else np.nan
    )

    rho_curve_corr_value = (
        float(np.nanmedian(rho_curve_corrs))
        if np.any(~np.isnan(rho_curve_corrs))
        else np.nan
    )

    rho_curve_bias_value = (
        float(np.nanmedian(rho_curve_biases))
        if np.any(~np.isnan(rho_curve_biases))
        else np.nan
    )

    rho_curve_abs_bias_value = (
        float(np.nanmedian(rho_curve_abs_biases))
        if np.any(~np.isnan(rho_curve_abs_biases))
        else np.nan
    )

    rho_curve_variance_value = (
        float(np.nanmedian(rho_curve_variances))
        if np.any(~np.isnan(rho_curve_variances))
        else np.nan
    )

    return {
        "ci_width_median": float(np.median(ci_widths)),
        "ci_width_mean": float(np.mean(ci_widths)),
        "bias_median": float(np.median(estimates - alpha_trues)),
        "abs_bias_median": float(np.median(np.abs(estimates - alpha_trues))),
        "coverage": float(np.mean(covered)),
        "alpha_true_mean": float(np.mean(alpha_trues)),
        "alpha_true_sd": float(np.std(alpha_trues)),
        "pool_total_median": float(np.median(pool_totals)),
        "rho_error_corr_median": rho_error_corr_value,
        "rho_curve_mse": rho_curve_mse_value,
        "rho_curve_corr": rho_curve_corr_value,
        "rho_curve_bias": rho_curve_bias_value,
        "rho_curve_abs_bias": rho_curve_abs_bias_value,
        "rho_curve_variance": rho_curve_variance_value,
    }


def run_one_setup(args):

    n_repositories, n_audit, err_mean, err_sd, change_sd, rho_scenario, design_tuple = (
        args
    )

    design, J, lambda_mix = design_tuple

    local_rng = np.random.default_rng(
        int(
            hashlib.md5(
                f"{n_repositories}_{n_audit}_{err_mean}_{err_sd}_{change_sd}_{rho_scenario}_{design}_{J}".encode()
            ).hexdigest()[:8],
            16,
        )
    )

    res = simulate_design(
        repository_error_rate_mean=err_mean,
        repository_error_rate_sd=err_sd,
        change_error_sd=change_sd,
        rho_scenario=rho_scenario,
        n_repositories=n_repositories,
        n_audit=n_audit,
        n_replicates=N_REPLICATES,
        design=design,
        J=J,
        lambda_mix=lambda_mix,
        bimodal_split=None,
        rng=local_rng,
    )

    return {
        "design": f"{design}_J{J}",
        "design_type": design,
        "J": J,
        "lambda": lambda_mix,
        "rho_scenario": rho_scenario,
        "n_repositories": n_repositories,
        "n_audit": n_audit,
        "repository_error_rate_mean": round(err_mean, 3),
        "repository_error_rate_sd": round(err_sd, 3),
        "change_error_sd": round(change_sd, 3),
        **res,
    }


results = Parallel(n_jobs=10, backend="loky")(
    delayed(run_one_setup)(setup) for setup in tqdm(setups)
)

df = pd.DataFrame(results)
df.head()

In [91]:
import pandas as pd
import numpy as np
import hashlib

# Selected configuration
SELECTED = {
    "design": "hybrid",
    "J": 3,
    "lambda_mix": 0.5,
    "n_audit": 100,
    "n_repositories": 5,
}

# Baseline + bimodal-Stärken
bimodal_scenarios = {
    "baseline": None,
    "mild (0.20/0.40)": (0.20, 0.40, 0.5),
    "strong (0.10/0.50)": (0.10, 0.50, 0.5),
}

stress_results = []

for bimodal_label, bimodal in bimodal_scenarios.items():
    for rho_scenario in scenarios.keys():
        seed = int(
            hashlib.md5(f"stress_{bimodal_label}_{rho_scenario}".encode()).hexdigest()[
                :8
            ],
            16,
        )
        rng = np.random.default_rng(seed)

        res = simulate_design(
            repository_error_rate_mean=0.30,  # ignored if bimodal
            repository_error_rate_sd=0.10,
            change_error_sd=0.10,
            rho_scenario=rho_scenario,
            n_repositories=SELECTED["n_repositories"],
            n_audit=SELECTED["n_audit"],
            n_replicates=200,
            design=SELECTED["design"],
            J=SELECTED["J"],
            lambda_mix=SELECTED["lambda_mix"],
            bimodal_split=bimodal,
            rng=rng,
        )

        stress_results.append(
            {
                "bimodal": bimodal_label,
                "scenario": rho_scenario,
                "coverage": res["coverage"],
                "ci_width": res["ci_width_median"],
                "bias": res["rho_curve_abs_bias"],
                "spread": np.sqrt(res["rho_curve_variance"]),
            }
        )

stress_df = pd.DataFrame(stress_results)

# Worst-case über Szenarien pro bimodal-Stärke
stress_summary = (
    stress_df.groupby("bimodal")
    .agg(
        coverage_worst=("coverage", "min"),
        ci_width_worst=("ci_width", "max"),
        bias_worst=("bias", "max"),
        spread_worst=("spread", "max"),
    )
    .reset_index()
)

print("Stress-Test Worst-Case per bimodal level:")
print(stress_summary.to_string(index=False))

Stress-Test Worst-Case per bimodal level:
           bimodal  coverage_worst  ci_width_worst  bias_worst  spread_worst
          baseline            0.97        0.181632    0.052044      0.048648
  mild (0.20/0.40)            0.96        0.184362    0.054905      0.046768
strong (0.10/0.50)            0.95        0.184721    0.053486      0.046976


In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

pio.templates["thesis"] = go.layout.Template(
    layout=go.Layout(
        font=dict(family="TeX Gyre Heros", size=12, color="black"),
        plot_bgcolor="white",
        paper_bgcolor="white",
        colorway=[
            "#0072B2",  # blue
            "#009E73",  # green
            "#E69F00",  # orange
            "#CC79A7",  # purple
            "#56B4E9",  # sky
            "#D55E00",  # vermillion
            "#F0E442",  # yellow (avoid for lines)
            "#000000",  # black
        ],
        colorscale=dict(
            sequential="Viridis",
            diverging="RdBu",
        ),
    )
)

pio.templates.default = "thesis"

In [111]:
import plotly.express as px

cov = df.groupby(
    [
        "n_repositories",
        "rho_scenario",
        "repository_error_rate_sd",
        "design_type",
        "n_audit",
    ],
    as_index=False,
)["coverage"].median()

fig = px.line(
    cov,
    x="n_audit",
    y="coverage",
    color="design_type",
    line_dash="repository_error_rate_sd",
    facet_row="rho_scenario",
    facet_col="n_repositories",
    markers=True,
    title="Coverage: repositories × rho × sigma",
)

fig.add_hline(y=0.95, line_dash="dash")
fig.update_layout(height=800, plot_bgcolor="white", paper_bgcolor="white")
fig.show()

ci = df.groupby(
    [
        "n_repositories",
        "rho_scenario",
        "repository_error_rate_sd",
        "design_type",
        "n_audit",
    ],
    as_index=False,
)["ci_width_median"].median()

px.line(
    ci,
    x="n_audit",
    y="ci_width_median",
    color="design_type",
    line_dash="repository_error_rate_sd",
    facet_row="rho_scenario",
    facet_col="n_repositories",
    markers=True,
    title="CI width: repositories × rho × sigma",
).update_layout(
    height=800,
    plot_bgcolor="white",
    paper_bgcolor="white",
).show()

rho_eval = df.groupby(
    [
        "n_repositories",
        "rho_scenario",
        "repository_error_rate_sd",
        "design_type",
        "n_audit",
    ],
    as_index=False,
)["rho_curve_mse"].median()

px.line(
    rho_eval,
    x="n_audit",
    y="rho_curve_mse",
    color="design_type",
    line_dash="repository_error_rate_sd",
    facet_row="rho_scenario",
    facet_col="n_repositories",
    markers=True,
    title="ρ diagnostic error (reconstruction of E[Y | ρ])",
).update_layout(
    height=800,
    plot_bgcolor="white",
    paper_bgcolor="white",
).show()

rho_corr = df.groupby(
    [
        "n_repositories",
        "rho_scenario",
        "repository_error_rate_sd",
        "design_type",
        "n_audit",
    ],
    as_index=False,
)["rho_curve_corr"].median()

px.line(
    rho_corr,
    x="n_audit",
    y="rho_curve_corr",
    color="design_type",
    line_dash="repository_error_rate_sd",
    facet_row="rho_scenario",
    facet_col="n_repositories",
    markers=True,
    title="ρ diagnostic correlation (shape of E[Y | ρ])",
).update_layout(
    height=800,
    plot_bgcolor="white",
    paper_bgcolor="white",
).show()

In [110]:
import plotly.express as px
import numpy as np

color_map = {
    "proportional": "#0072B2",  # Okabe-Ito blue
    "equal": "#009E73",  # Okabe-Ito bluish green
    "hybrid λ=0.25": "#F0E442",  # Okabe-Ito yellow
    "hybrid λ=0.50": "#E69F00",  # Okabe-Ito orange
    "hybrid λ=0.75": "#D55E00",  # Okabe-Ito vermillion
}

df_plot = df[
    (df["repository_error_rate_mean"] == 0.50)
    & (df["repository_error_rate_sd"] == 0.10)
    & (df["change_error_sd"] == 0.10)
    & (df["n_repositories"] == 5)
    & (df["J"] == 3)
].copy()

df_plot["rho_se"] = np.sqrt(df_plot["rho_curve_variance"])
df_plot["rho_scenario"] = df_plot["rho_scenario"].replace(
    {"Non-Monotonic": "Decoupled"}
)


def design_label(row):
    if row["design_type"] == "hybrid":
        return f"hybrid λ={row['lambda']:.2f}"
    return row["design_type"]


df_plot["Design"] = df_plot.apply(design_label, axis=1)

df_plot = df_plot.groupby(
    ["rho_scenario", "Design", "n_audit"],
    as_index=False,
).agg(
    CI_Width=("ci_width_median", "median"),
    Bias=("rho_curve_abs_bias", "median"),
)

df_plot = df_plot.rename(columns={"CI_Width": "CI Width", "n_audit": "n"})


df_long = df_plot.melt(
    id_vars=["n", "Design", "rho_scenario"],
    value_vars=["CI Width", "Bias"],
    var_name="Metric",
    value_name="Value",
)

design_order = [
    "proportional",
    "equal",
    "hybrid λ=0.25",
    "hybrid λ=0.50",
    "hybrid λ=0.75",
]

scenario_order = ["Monotonic", "Decoupled", "Uninformative"]

fig = (
    px.line(
        df_long,
        x="n",
        y="Value",
        color="Design",  # statt line_dash
        color_discrete_map=color_map,
        category_orders={"Design": design_order, "rho_scenario": scenario_order},
        facet_row="rho_scenario",
        facet_col="Metric",
        facet_col_spacing=0.03,
        facet_row_spacing=0.06,
    )
    .update_traces(mode="lines", line=dict(width=1))
    .update_yaxes(
        showticklabels=True,
        matches=None,
        automargin=True,
        title_text="",
        nticks=3,
    )
    .update_xaxes(title_font=dict(size=12), title_standoff=5)
    .for_each_annotation(
        lambda a: a.update(
            text=a.text.replace("Metric=", "").replace("rho_scenario=", "")
        )
    )
    .update_layout(
        height=500,
        width=800,
        plot_bgcolor="white",
        paper_bgcolor="white",
        legend_title_text="",
        legend=dict(
            orientation="h",
            yanchor="top",
            y=-0.1,
            xanchor="right",
            x=1.03,
        ),
        margin=dict(l=2, r=0, t=40, b=2),
    )
)

for a in fig.layout.annotations:
    if a.text in ["Monotonic", "Decoupled", "Uninformative"]:
        a.x += 0.01

for a in fig.layout.annotations:
    if a.text in ["CI Width", "Bias", "Spread"]:
        a.yshift = 10

for xaxis in fig.select_xaxes():
    d0, d1 = xaxis.domain
    xaxis.domain = [d0 + 0.02, d1 - 0.01]

metric_order = df_long["Metric"].unique().tolist()


fig.show(
    config={
        "toImageButtonOptions": {
            "format": "svg",
            "filename": "plot",
            "height": 500,
            "width": 800,
            "scale": 1,
        }
    }
)

In [ ]:
import pandas as pd
import numpy as np

design_keys = ["design_type", "J", "lambda", "n_audit", "n_repositories"]

agg = (
    df.groupby(design_keys, dropna=False)
    .agg(
        coverage_worst=("coverage", "min"),
        ci_width_worst=("ci_width_median", "max"),
        bias_worst=("rho_curve_abs_bias", "max"),
        variance_worst=("rho_curve_variance", "max"),
    )
    .reset_index()
)

agg["cost"] = agg["n_audit"] * agg["n_repositories"]

COVERAGE_MIN = 0.92
CI_WIDTH_MAX = 0.25
BIAS_MAX = 0.08
VARIANCE_MAX = 0.005

feasible = (
    agg[
        (agg["coverage_worst"] >= COVERAGE_MIN)
        & (agg["ci_width_worst"] <= CI_WIDTH_MAX)
        & (agg["bias_worst"] <= BIAS_MAX)
        & (agg["variance_worst"] <= VARIANCE_MAX)
    ]
    .copy()
    .reset_index(drop=True)
)

print(f"Total designs: {len(agg)}")
print(f"Feasible designs (coverage >= {COVERAGE_MIN}): {len(feasible)}")


def pareto_mask(df, minimize_cols):
    arr = df[minimize_cols].values
    n = len(arr)
    is_pareto = np.ones(n, dtype=bool)
    for i in range(n):
        if not is_pareto[i]:
            continue
        for j in range(n):
            if i == j:
                continue
            if np.all(arr[j] <= arr[i]) and np.any(arr[j] < arr[i]):
                is_pareto[i] = False
                break
    return is_pareto


minimize_cols = ["ci_width_worst", "bias_worst", "variance_worst", "cost"]
feasible["is_pareto"] = pareto_mask(feasible, minimize_cols)
pareto_front = feasible[feasible["is_pareto"]].copy()

selected = pareto_front.loc[pareto_front["cost"].idxmin()]

print(f"\nPareto front size: {len(pareto_front)}")
print("\nPareto front sorted by cost:")
display_cols = design_keys + [
    "coverage_worst",
    "ci_width_worst",
    "bias_worst",
    "variance_worst",
    "cost",
]
print(pareto_front.sort_values("cost")[display_cols].to_string(index=False))

print("\nSelected configuration:")
print(selected[display_cols])

Total designs: 150
Feasible designs (coverage >= 0.92): 6

Pareto front size: 4

Pareto front sorted by cost:
design_type  J  lambda  n_audit  n_repositories  coverage_worst  ci_width_worst  bias_worst  variance_worst  cost
     hybrid  3    0.75       80               5           0.920        0.209483    0.072755        0.004493   400
     hybrid  3    0.50      100               5           0.925        0.203156    0.058680        0.002762   500
     hybrid  3    0.75      100               5           0.925        0.187660    0.062790        0.003204   500
     hybrid  3    0.50      100              10           0.925        0.202645    0.061929        0.003235  1000

Selected configuration:
design_type         hybrid
J                        3
lambda                0.75
n_audit                 80
n_repositories           5
coverage_worst        0.92
ci_width_worst    0.209483
bias_worst        0.072755
variance_worst    0.004493
cost                   400
Name: 3, dtype: object
